# Day 12 | ILT 1: Introduction to Genie & Semantic Layer
### GlobalMart Data Engineering Bootcamp

| | |
|---|---|
| **Calendar slot** | Day 12, 11:00 AM – 1:00 PM (ILT) |
| **Duration** | 120 minutes |
| **Format** | Concept ILT + real, runnable semantic-layer prep code — Genie Agent setup itself is not scripted here (see "What's Next") |
| **Builds on** | Day 7 HOL 2 (`gbmart.gold.fact_sales`, `vw_monthly_category_sales`, `vw_regional_sales`), Day 6 ILT 3 (measure additivity rules) |
| **Module status** | Standalone module — listed separately from the main day-by-day GlobalMart pipeline build, per the course calendar |

### Learning Objectives
- Explain what Databricks Genie is and how business users query it in plain English
- Explain what a semantic layer is, and why an LLM (or a business user) cannot make sense of raw column names without one
- Identify what makes a semantic layer "good" for Genie: comments, certified views, unambiguous names, sample questions
- Write real `COMMENT ON TABLE` / `ALTER TABLE ... ALTER COLUMN ... COMMENT` statements against a safe practice copy of `fact_sales`, and verify they landed
- Recognize the honest boundary of this session: Genie Agent setup itself is UI-configured, not something a notebook can script

---
**Instructor note:** This is a genuinely standalone module on the calendar — unlike most days in this course, it does not chain forward into the next day's build. The matching hands-on, *"Set Up Genie on Gold Layer — Descriptions + Natural Language Queries,"* is a UI-driven Databricks Workspace exercise (clicking through Catalog Explorer / the Genie UI to create a Genie Agent) and is deliberately deferred to a later, separate session against the real `gbmart` Gold layer. Everything in this notebook that can honestly be scripted, is. The one thing that can't — creating the Genie Agent itself — is called out plainly instead of faked with code that wouldn't actually work.

---
**Safety note:** This notebook only ever *reads* from `gbmart.gold.fact_sales` — one `SELECT ... LIMIT 100`. Every `COMMENT` / `ALTER TABLE` statement below runs against your own practice copy, never the real table. Nothing here starts a job, workflow, pipeline, SQL warehouse, or cluster.

**Instructions:** Run each cell in order with **Shift + Enter**.

---
## Section 1 — What Is Databricks Genie?

**Genie** is a Databricks AI/BI feature that lets a business user type a plain-English question — *"What was our revenue by category last month?"* — against a curated set of tables and views called a **Genie Agent**, and get back a generated SQL query plus a tabular/plotted answer. The person asking never writes SQL.

> **Naming note, worth reading yourself:** Databricks' own docs (`docs.databricks.com/aws/en/genie/set-up`) state directly: *"Genie Agents were formerly known as Genie Spaces."* If your workspace's UI still says "Genie Space" somewhere, that's expected — feature renames roll out to different accounts on different schedules. This course uses "Genie Agent" going forward since that's what the current docs say, but don't be thrown if you see the older name live.

### What actually happens behind the scenes
1. A business user types a natural-language question into the Genie chat UI.
2. Genie's underlying LLM looks at the Genie Agent's **metadata** — table names, column names, comments, sample questions — to work out which tables, columns, and joins answer the question.
3. Genie generates a SQL query, runs it against a SQL warehouse, and returns the result — usually along with the SQL itself, so a data engineer can verify exactly what ran.

### What a Genie Agent is
A **Genie Agent** is a scoped, curated subset of a catalog — specific tables and views, plus instructions, sample questions, and example SQL — that someone sets up so Genie has a bounded, well-understood surface to translate questions against. It is deliberately narrower than "the whole catalog," so the LLM is not guessing across hundreds of unrelated tables.

### Be honest about the boundary
**A Genie Agent is set up entirely by clicking through the Databricks workspace UI** — Genie Agents in the sidebar → New → choosing data sources → Create, then writing instructions and adding sample questions — **not by running code in a notebook.** There is no cell later in this notebook that creates a Genie Agent, and there will not be one. That part of *"Set Up Genie on Gold Layer"* is a separate, later hands-on session, done live in the Databricks workspace UI against the real `gbmart` Gold layer.

---
## Section 2 — What a Semantic Layer Is, and Why Genie Needs One

Look at two raw facts about `gbmart.gold.fact_sales`:

- A column named `Sales_amount`, type `decimal`
- A column named `Actual_price`, type `decimal`

To an LLM — or a brand-new analyst who has never seen this schema — those two names alone answer none of the questions that actually matter:
- Is `Sales_amount` the line item's revenue, or the whole order's revenue?
- Is it safe to `SUM(Actual_price)` to get "total sales," or will that silently produce a meaningless number?
- Pre-discount or post-discount? What currency?

**A semantic layer is the metadata that answers these questions before anyone has to ask.** Concretely, for this course, it is:
- **Table comments** — what this table is, at what grain, how it's populated
- **Column comments** — what a specific column means in business language, including whether it is safe to aggregate
- **Certified views with business-friendly names** — a pre-built, pre-agreed answer to a common question, so nobody (human or Genie) reconstructs the same join and aggregation logic from scratch every time
- **Sample questions and example SQL** registered against a Genie Agent, so Genie has worked examples to pattern-match against

None of this changes what the data *is*. All of it changes whether a raw schema is *usable* by something that has never seen your business before — which is exactly the position both Genie's LLM and a new hire are in on day one.

### Why Genie specifically depends on this
Genie's natural-language-to-SQL step is only as good as the metadata it can see. Ask Genie "what's our revenue by category last month" against a Genie Agent with no comments and cryptically named tables, and it has to guess which table has "category" and which column is actually "revenue." Ask the same question against a Genie Agent where `fact_sales.Sales_amount` is commented *"Revenue in INR for this order line item — safe to SUM"* and a certified view is literally named `vw_monthly_category_sales`, and Genie's job becomes dramatically easier: it is matching a well-described question against well-described tables, not reverse-engineering your business from column names alone.

---
## Section 3 — What Makes a Good Semantic Layer for Genie

| Ingredient | What it looks like | GlobalMart example |
|---|---|---|
| Clear table comments | Plain business language: what the table represents, its grain | "One row per order line item — GlobalMart's core sales fact table" |
| Clear column comments | What the column means, and whether it's safe to aggregate | `Sales_amount`: "Revenue in INR ... safe to SUM." `Actual_price`: "... non-additive, never SUM directly." |
| Certified views, descriptive names | Pre-built answers to the questions people actually ask, named so the question is obvious | `vw_monthly_category_sales`, `vw_regional_sales` (built Day 7 HOL 2) |
| Unambiguous column names | Avoid cryptic abbreviations or overloaded names | GlobalMart spells out `Discounted_price`, not `disc_prc` |
| Sample questions + example SQL | A handful of worked question → SQL pairs registered on the Genie Agent | See Section 5 below |

### GlobalMart already has two semantic-layer artifacts — built before Genie ever entered the conversation
Day 7 HOL 2 built two real Gold views on top of `fact_sales`:

- **`gbmart.gold.vw_monthly_category_sales`** — year/month/category/sub_category revenue, quantity, order count, and average discount given. The name alone answers "what question does this view answer?"
- **`gbmart.gold.vw_regional_sales`** — state/city revenue, order count, customer count, and average order value.

Neither view was built with Genie in mind — they exist because category managers and regional ops leads needed one agreed-upon number to report against (Day 7 HOL 2's own reasoning: *"one agreed-upon definition everyone reports against, instead of every analyst writing a slightly different join"*). **That is precisely what a semantic layer is.** A Genie Agent built on `gbmart.gold` would point at these two views directly for exactly the questions they already answer, instead of asking Genie to reconstruct the same multi-table join and `GROUP BY` from raw `fact_sales`, every single time.

---
## Section 4 — The Part We CAN Script: Preparing Semantic-Layer Metadata

Genie Agent setup is UI-only (Section 1). But the semantic layer *underneath* a Genie Agent — table comments, column comments, and verifying they landed — is completely real, scriptable Databricks SQL. That is what the rest of this notebook does, against a **safe practice copy**, never the real `gbmart.gold.fact_sales` table.

**What we will do, in order:**
1. Create your own practice schema
2. Copy 100 rows of the real `fact_sales` shape into it — read-only against the real table, one `SELECT ... LIMIT 100`
3. Look at that copy's schema with **zero** comments — exactly what an LLM sees with no semantic layer
4. Add a real table comment and real column comments, in business language, matching Day 6 ILT 3's additivity rules exactly
5. Look at the same schema again, now with comments, and compare

**What we will NOT do:** call any Genie API or SDK, or otherwise attempt to create a Genie Agent from code. As of this course's design, that setup is UI-only — faking it with code that would not actually work would teach the wrong lesson.

In [ ]:
# --- Practice schema setup -- replace YOUR_SCHEMA with something unique to you, ---
# --- e.g. main.priya_genie_lab (same pattern as Day 10's SCD2 practice schema)  ---
YOUR_SCHEMA = "main.YOUR_SCHEMA"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {YOUR_SCHEMA}")

PRACTICE_TABLE = f"{YOUR_SCHEMA}.fact_sales_practice"

print(f"Practice schema ready : {YOUR_SCHEMA}")
print(f"Practice table target : {PRACTICE_TABLE}")

### Seeding the practice copy — read-only against the real table

The only touch against the real `gbmart.gold.fact_sales` table anywhere in this notebook is the `SELECT ... LIMIT 100` below. It is read-only — no `ALTER`, no `COMMENT`, nothing that changes the real table's data or metadata. `CREATE OR REPLACE TABLE ... AS SELECT` (CTAS) also makes this cell safe to re-run: running it again just rebuilds the same 100-row practice copy from scratch, it does not accumulate duplicate rows or fail on a second run.

In [ ]:
# CTAS: a small, independent practice copy -- 100 rows, same column shape as the
# real fact_sales, zero inherited comments (CTAS does not copy column/table
# comments over). CREATE OR REPLACE makes this cell idempotent -- safe to re-run.
ctas_sql = (
    f"CREATE OR REPLACE TABLE {PRACTICE_TABLE} "
    f"AS SELECT * FROM gbmart.gold.fact_sales LIMIT 100"
)
spark.sql(ctas_sql)

practice_row_count = spark.table(PRACTICE_TABLE).count()
print(f"Practice copy created : {PRACTICE_TABLE}")
print(f"Rows                  : {practice_row_count}")
print("Read-only sample of the real table -- nothing was written back to gbmart.gold.")

### Step A — What This Schema Looks Like With Zero Semantic Layer

This is exactly what an LLM (or a brand-new analyst) sees before any comments exist: a list of names and data types, and nothing else.

In [ ]:
# "Before" -- raw column names and types only, no business meaning attached.
# This dtypes list is literally all the schema information available right now --
# no comments exist yet on this freshly-CTAS'd practice table.
print("=== RAW SCHEMA -- what an LLM sees with NO semantic layer ===")
print()
for col_name, col_type in spark.table(PRACTICE_TABLE).dtypes:
    print(f"  {col_name:<22} {col_type}")

print()
print("Questions an LLM cannot answer from the above alone:")
print("  Is it safe to SUM(Actual_price)? Is Sales_amount pre- or post-discount? What currency?")

---
### Step B — Add Real Semantic-Layer Metadata

Now we add a table comment and per-column comments, in plain business language. These comments **must stay consistent with Day 6 ILT 3's measure additivity rules** — getting this wrong in a semantic layer is worse than having no comment at all, because Genie (and every analyst downstream) will trust it:

| Measure | Additivity (Day 6 ILT 3) | What the comment must say |
|---|---|---|
| `Quantity_purchased` | Additive | Safe to SUM across any dimension |
| `Sales_amount` | Additive | Safe to SUM — this is the revenue measure |
| `Actual_price` | **Non-additive** | Never SUM/AVG directly across products |
| `Discounted_price` | **Non-additive** | Never SUM/AVG directly across products; derive avg. selling price as `SUM(Sales_amount) / SUM(Quantity_purchased)` |

In [ ]:
# Real, runnable table comment -- on the PRACTICE copy only. Never run this
# kind of statement against the real gbmart.gold table.
table_comment = (
    "Practice copy (100-row sample) of the real GlobalMart fact_sales Gold "
    "table -- one row per order line item. Built for the Day 12 Genie / "
    "semantic-layer session. Not a production table -- do not point a real "
    "dashboard or Genie Agent at this table."
)

spark.sql(f"COMMENT ON TABLE {PRACTICE_TABLE} IS '{table_comment}'")
print(f"Table comment set on {PRACTICE_TABLE}")

In [ ]:
# Column comments -- one per fact_sales column, written in plain business
# language. Additivity wording matches Day 6 ILT 3 exactly -- do not contradict it.
column_comments = {
    "fact_sales_sk": (
        "Surrogate primary key for this fact table, generated as "
        "sha2(order_item_id, 256). Not a business identifier -- never expose "
        "this as a lookup value to end users."
    ),
    "Payment_ID": (
        "Natural key identifying the payment record for this order line item "
        "(source: silver.payments). One payment per order. Does not resolve "
        "to dim_payment_method -- different ID space entirely."
    ),
    "Customer_ID": (
        "Natural key identifying the customer who placed this order line "
        "item. Joins to dim_customer on the business key, not the "
        "customer_sk surrogate key."
    ),
    "Product_ID": (
        "Natural key identifying the product sold in this order line item. "
        "Joins to dim_product filtered to is_current = true."
    ),
    "Order_ID": (
        "Natural key identifying the parent order this line item belongs "
        "to. Use COUNT(DISTINCT Order_ID) to count orders -- COUNT(*) "
        "counts line items instead."
    ),
    "Address_ID": (
        "Natural key identifying the delivery address for this order line "
        "item (one primary address per customer)."
    ),
    "Time_ID": (
        "Date key (YYYYMMDD integer) identifying when this order line item "
        "was placed. Joins to dim_date.date_key."
    ),
    "Quantity_purchased": (
        "Number of units purchased in this order line item. Additive -- "
        "safe to SUM across any dimension (product, customer, date, "
        "category, region)."
    ),
    "Actual_price": (
        "Original list price per unit, in INR, before discount. "
        "Non-additive -- a rate, not a quantity. Never SUM or AVG this "
        "directly across products; use only combined with Discounted_price "
        "for discount-gap analysis."
    ),
    "Discounted_price": (
        "Price per unit actually charged, in INR, after discount. "
        "Non-additive -- never SUM or AVG this directly across products. "
        "Derive average selling price as SUM(Sales_amount) / "
        "SUM(Quantity_purchased) instead."
    ),
    "Sales_amount": (
        "Revenue in INR for this order line item -- quantity times "
        "discounted price. Additive and safe to SUM across any dimension; "
        "this is the primary revenue measure."
    ),
}

for column_name, comment_text in column_comments.items():
    spark.sql(f"ALTER TABLE {PRACTICE_TABLE} ALTER COLUMN {column_name} COMMENT '{comment_text}'")

print(f"Set {len(column_comments)} column comments on {PRACTICE_TABLE}")

### Step C — Verify the Comments Landed

In [ ]:
# "After" -- same table, now with real table + column comments landed.
# DESCRIBE TABLE returns col_name / data_type / comment for every column.
print("=== SCHEMA WITH SEMANTIC LAYER -- what Genie/an LLM sees now ===")
print()
described = spark.sql(f"DESCRIBE TABLE {PRACTICE_TABLE}").collect()
for row in described:
    print(f"  {row['col_name']:<22} {row['data_type']:<10} {row['comment']}")

# The table-level comment lives in DESCRIBE TABLE EXTENDED's "Comment" row.
extended_df = spark.sql(f"DESCRIBE TABLE EXTENDED {PRACTICE_TABLE}")
table_comment_row = extended_df.filter(extended_df.col_name == "Comment").collect()
print()
if table_comment_row:
    print(f"Table comment: {table_comment_row[0]['data_type']}")

---
### Before vs. After — Why This Is the Whole Point

| | Before (Step A) | After (Step C) |
|---|---|---|
| What a column name tells you | `Actual_price`, `decimal` — and nothing else | Same column, plus: "non-additive, never SUM this directly across products" |
| Can Genie safely answer "total sales"? | Has to guess whether to sum `Actual_price`, `Discounted_price`, or `Sales_amount` | Knows `Sales_amount` is the additive revenue measure — the comment says so directly |
| Can a new analyst self-serve? | Only by asking someone who already knows the business rules | Yes — the rule is attached to the column itself, in `DESCRIBE TABLE` output |

Nothing about the underlying data changed between Step A and Step C — same 100 rows, same values. **The only thing that changed is whether the schema can explain itself.** That is the entire value of a semantic layer, and it is also exactly what Genie's natural-language-to-SQL translation leans on.

---
## Section 5 — Sample Questions: Bootstrapping Genie's Natural-Language Matching

A Genie Agent lets you register a handful of **sample questions**, each paired with the SQL that correctly answers it. Genie uses these as worked examples — when a real user's question looks similar to a registered sample, Genie leans on that known-good pattern instead of generating a join from scratch. The table below is illustrative only: it shows what you would type into the Genie Agent UI in the later hands-on session. It is not executed here.

| Sample question | Maps to |
|---|---|
| "What was our revenue by category last month?" | `SELECT category, SUM(total_revenue) FROM gbmart.gold.vw_monthly_category_sales WHERE year = ... AND month = ... GROUP BY category` |
| "Which states generate the most revenue?" | `SELECT state, SUM(total_revenue) FROM gbmart.gold.vw_regional_sales GROUP BY state ORDER BY total_revenue DESC` |
| "What is our average discount by category?" | `SELECT category, AVG(avg_discount_given) FROM gbmart.gold.vw_monthly_category_sales GROUP BY category` |
| "How many orders came from each region?" | `SELECT state, city, total_orders FROM gbmart.gold.vw_regional_sales` |

Notice every sample question maps to one of the two **already-built** Gold views from Day 7 HOL 2 — not to a fresh query against raw `fact_sales`. That is the point of a semantic layer: Genie's easiest, most reliable answers come from certified views someone already got right, not from re-deriving the join and aggregation logic live, per question.

---
## Key Takeaways

1. **Genie translates natural language to SQL against a curated Genie Agent** — business users ask questions in plain English, Genie generates and runs the SQL, and (usually) shows its work.
2. **Genie Agents are set up entirely through the Databricks workspace UI** — there is no SDK/API call in this notebook that creates one, because that is not how this course (or Databricks, as of this design) does it.
3. **A semantic layer is what makes raw tables usable by something that has never seen your business before** — table comments, column comments, certified views, unambiguous names, and sample questions.
4. **`vw_monthly_category_sales` and `vw_regional_sales` (Day 7 HOL 2) are already real semantic-layer artifacts** — built before Genie was ever discussed, still exactly what Genie needs.
5. **Comments must be additivity-correct, or they actively mislead** — `Quantity_purchased`/`Sales_amount` are safe to SUM; `Actual_price`/`Discounted_price` are not (Day 6 ILT 3). A wrong comment is worse than no comment.
6. **Every `COMMENT` / `ALTER TABLE` statement in this notebook ran against a 100-row practice copy** — `gbmart.gold.fact_sales` itself was only ever read, once, via `LIMIT 100`.

### Self-Check
- [ ] I can explain, in one sentence, what a Genie Agent is and how it differs from "the whole catalog"
- [ ] I can name at least 3 ingredients of a good semantic layer
- [ ] I can state which 2 of `fact_sales`'s 4 measures are additive, and which 2 are not, without looking it up
- [ ] I ran the practice-copy `COMMENT` / `ALTER TABLE` cells and saw the "after" schema show real comments where the "before" schema showed none
- [ ] I can explain why this notebook never calls a Genie API, and where the real Genie Agent setup will happen instead

### What's Next
The matching hands-on, **"Set Up Genie on Gold Layer — Descriptions + Natural Language Queries,"** is a separate, later session done live in the Databricks workspace UI against the real `gbmart.gold` layer: creating an actual Genie Agent over `fact_sales` and its dimensions/views, adding the descriptions and sample questions this session prepared you to write, and asking it real natural-language questions. Nothing about that session can be scripted in a notebook — that is *why* it is deferred and separate, not a gap in this one.

---
## Reset (if needed)

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {PRACTICE_TABLE}")
# print(f"Reset complete -- {PRACTICE_TABLE} dropped. gbmart.gold.fact_sales was never touched.")